# Ch.9 — ML Experiment Tracking & Model Registry (Azure Supplement)

**Azure-specific extension:** This notebook adapts the main `notebook.ipynb` concepts to Azure ML infrastructure. Instead of local MLflow tracking, we explore:

- **Azure ML Workspace** connection and experiment logging
- **Azure ML HyperDrive** for distributed hyperparameter sweeps
- **Azure ML Model Registry** for versioned model storage
- **Azure Cost Tracking** for experiment budgeting

**Prerequisites:**
- Complete the main `notebook.ipynb` first (understands MLflow, experiment tracking, model registry)
- Azure subscription (Free Tier sufficient for demo — uses compute clusters for training)
- Azure ML SDK installed: `pip install azureml-core azureml-sdk`

**What you'll build:**
1. Azure ML workspace connection (credential-based or managed identity)
2. Experiment logging with Azure ML Run API
3. HyperDrive hyperparameter sweep (distributed across compute cluster)
4. Model registry workflow (register, version, download)
5. Cost tracking dashboard (per-experiment compute costs)
6. Comparison: Azure ML vs. MLflow UI

**Running example:** Same BERT fine-tuning from main notebook, but scaled to Azure compute.

In [ ]:
# ── Azure ML Credentials Setup ───────────────────────────────────────────────
# USER: Replace with your Azure ML credentials before running live experiments.
# These will be stripped by the pre-push hook — NEVER commit real credentials.

AZURE_SUBSCRIPTION_ID = ""  # Your Azure subscription ID (leave empty to skip live API calls)
AZURE_RESOURCE_GROUP = ""   # Resource group containing Azure ML workspace
AZURE_WORKSPACE_NAME = ""   # Azure ML workspace name
AZURE_REGION = "eastus"     # Default region (change to westus2, westeurope, etc.)

# Optional: Azure ML compute cluster name (for HyperDrive sweeps)
AZURE_COMPUTE_CLUSTER = "gpu-cluster"  # Will create if doesn't exist
AZURE_COMPUTE_VM_SIZE = "Standard_NC6s_v3"  # 1×V100, 6 vCPUs, $3.06/hr

# Note: If credentials are empty, notebook will use mock data for demonstration.
# To run live experiments, fill in credentials and ensure Azure CLI is authenticated:
#   az login
#   az account set --subscription <AZURE_SUBSCRIPTION_ID>
#   az ml workspace create --name <WORKSPACE_NAME> --resource-group <RESOURCE_GROUP>

USE_LIVE_API = bool(AZURE_SUBSCRIPTION_ID and AZURE_RESOURCE_GROUP and AZURE_WORKSPACE_NAME)

if USE_LIVE_API:
    print(f"✓ Live Azure ML mode enabled")
    print(f"  Subscription: {AZURE_SUBSCRIPTION_ID[:8]}...")
    print(f"  Resource Group: {AZURE_RESOURCE_GROUP}")
    print(f"  Workspace: {AZURE_WORKSPACE_NAME}")
    print(f"  Region: {AZURE_REGION}")
else:
    print("⚠ Demo mode: using mock Azure ML data (no API calls)")
    print("  To enable live experiments, set credentials above.")
    print("  Required: AZURE_SUBSCRIPTION_ID, AZURE_RESOURCE_GROUP, AZURE_WORKSPACE_NAME")

In [ ]:
# ── Azure ML SDK Setup & Workspace Connection ─────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

# Azure ML SDK imports (install: pip install azureml-core azureml-sdk)
if USE_LIVE_API:
    try:
        from azureml.core import Workspace, Experiment, Run, ScriptRunConfig, Environment
        from azureml.core.compute import ComputeTarget, AmlCompute
        from azureml.train.hyperdrive import (
            HyperDriveConfig, GridParameterSampling, PrimaryMetricGoal,
            BanditPolicy, choice, uniform
        )
        from azureml.core.model import Model
        
        # Connect to Azure ML workspace
        ws = Workspace(
            subscription_id=AZURE_SUBSCRIPTION_ID,
            resource_group=AZURE_RESOURCE_GROUP,
            workspace_name=AZURE_WORKSPACE_NAME
        )
        
        print(f"✓ Connected to Azure ML workspace: {ws.name}")
        print(f"  Region: {ws.location}")
        print(f"  SDK version: {azureml.core.VERSION}")
        
    except ImportError:
        print("⚠ Azure ML SDK not installed. Install with:")
        print("  pip install azureml-core azureml-sdk")
        USE_LIVE_API = False
    except Exception as e:
        print(f"⚠ Azure ML workspace connection failed: {e}")
        print("  Verify credentials and workspace exists:")
        print("  az ml workspace show --name <WORKSPACE_NAME> --resource-group <RESOURCE_GROUP>")
        USE_LIVE_API = False

# Mock workspace for demo mode
if not USE_LIVE_API:
    print("\n🔧 Using mock workspace for demonstration")
    
    class MockWorkspace:
        name = "demo-workspace"
        location = "eastus"
        subscription_id = "demo-subscription"
    
    ws = MockWorkspace()

print("\nLibraries loaded. Ready.")

## 1 · Log Experiment to Azure ML

Azure ML experiments work similarly to MLflow, but with tighter integration to Azure compute and storage:

**Key differences from MLflow:**
- **Experiments** are organized in the Azure ML workspace (visible in Azure Portal)
- **Runs** are automatically tracked with compute metadata (VM type, region, duration)
- **Artifacts** (models, plots) are stored in Azure Blob Storage (workspace default storage)
- **Metrics** can trigger Azure Monitor alerts (e.g., "notify if accuracy < 0.9")

**Running example:** Log a baseline BERT fine-tuning run (same as MLflow notebook).

**Cost note:** Starting a `Run` without compute is free (just metadata logging). Actual training on Azure ML compute incurs charges.

In [ ]:
# ── Log Baseline Experiment (Local Run, No Compute) ──────────────────────────
# Similar to MLflow: log hyperparameters, metrics, artifacts
# This cell runs locally (no Azure compute charges) — just logs metadata to workspace

if USE_LIVE_API:
    # Create experiment (or get existing)
    experiment = Experiment(workspace=ws, name="bert-finetuning-baseline")
    
    # Start logging run
    run = experiment.start_logging()
    
    # Log hyperparameters
    run.log("learning_rate", 5e-5)
    run.log("batch_size", 32)
    run.log("epochs", 3)
    run.log("max_seq_length", 128)
    run.log("warmup_steps", 500)
    
    # Simulate training loop (mock metrics)
    for epoch in range(1, 4):
        # Mock metrics (in real scenario, these come from actual training)
        train_loss = 0.8 - (epoch * 0.15) + np.random.uniform(-0.05, 0.05)
        val_accuracy = 0.70 + (epoch * 0.08) + np.random.uniform(-0.02, 0.02)
        val_f1 = 0.68 + (epoch * 0.09) + np.random.uniform(-0.02, 0.02)
        
        run.log("train_loss", train_loss, step=epoch)
        run.log("val_accuracy", val_accuracy, step=epoch)
        run.log("val_f1", val_f1, step=epoch)
    
    # Final metrics
    run.log("final_accuracy", 0.87)
    run.log("final_f1", 0.86)
    
    # Complete run
    run.complete()
    
    print(f"✓ Logged baseline experiment: {run.id}")
    print(f"  View in Azure Portal: {run.get_portal_url()}")
    
else:
    print("Demo mode: Baseline experiment logged (mock)")
    print("  Hyperparameters: lr=5e-5, batch_size=32, epochs=3")
    print("  Final metrics: accuracy=0.87, f1=0.86")
    print("\n💡 In live mode, this would create an experiment in your Azure ML workspace.")
    print("   Run `az ml run list --experiment-name bert-finetuning-baseline` to see all runs.")

## 2 · Hyperparameter Sweep with Azure ML HyperDrive

**HyperDrive** = Azure ML's distributed hyperparameter tuning service

**Key features:**
- **Sampling strategies:** Grid, random, Bayesian
- **Early termination:** Bandit policy (stop underperforming runs early)
- **Distributed execution:** Run 10+ experiments in parallel on compute cluster
- **Cost optimization:** Terminate bad runs quickly, scale down cluster after sweep

**Running example:** Same 12-run grid search from MLflow notebook:
- Learning rate: [1e-5, 3e-5, 5e-5]
- Batch size: [16, 32]
- Warmup steps: [100, 500]

**Cost estimate (for reference):**
- VM: `Standard_NC6s_v3` (1×V100, $3.06/hr)
- Duration: ~20 min per run × 12 runs = 4 hours total
- Cost: $3.06/hr × 4 hrs = **$12.24** (if runs are sequential)
- With 4-node cluster (parallel): 1 hour × $3.06 × 4 = **$12.24** (same cost, faster)

**Note:** This cell demonstrates HyperDrive *configuration* only. Actual execution requires Azure ML compute cluster.

In [ ]:
# ── Azure ML HyperDrive Configuration (Grid Search) ──────────────────────────
# Define parameter space for hyperparameter sweep

if USE_LIVE_API:
    # Create or get existing compute cluster
    try:
        compute_target = ComputeTarget(workspace=ws, name=AZURE_COMPUTE_CLUSTER)
        print(f"✓ Using existing compute cluster: {AZURE_COMPUTE_CLUSTER}")
    except:
        print(f"Creating new compute cluster: {AZURE_COMPUTE_CLUSTER}...")
        compute_config = AmlCompute.provisioning_configuration(
            vm_size=AZURE_COMPUTE_VM_SIZE,
            min_nodes=0,  # Scale to zero when idle (cost savings)
            max_nodes=4,  # Max 4 parallel runs
            idle_seconds_before_scaledown=600  # 10 min idle before scale-down
        )
        compute_target = ComputeTarget.create(ws, AZURE_COMPUTE_CLUSTER, compute_config)
        compute_target.wait_for_completion(show_output=True)
    
    # Define parameter search space (grid search)
    param_sampling = GridParameterSampling({
        "learning_rate": choice([1e-5, 3e-5, 5e-5]),
        "batch_size": choice([16, 32]),
        "warmup_steps": choice([100, 500])
    })
    
    # Early termination policy (stop runs if accuracy not improving)
    early_termination_policy = BanditPolicy(
        evaluation_interval=1,  # Check every epoch
        slack_factor=0.1,       # Terminate if accuracy < (best_accuracy * 0.9)
        delay_evaluation=1      # Don't terminate in first epoch
    )
    
    # HyperDrive configuration
    # Note: Requires training script (e.g., train_bert.py) in workspace
    hyperdrive_config = HyperDriveConfig(
        run_config=ScriptRunConfig(
            source_directory="./training_scripts",  # Local directory with train_bert.py
            script="train_bert.py",
            compute_target=compute_target,
            environment=Environment.from_pip_requirements(
                name="bert-env",
                file_path="requirements.txt"
            )
        ),
        hyperparameter_sampling=param_sampling,
        policy=early_termination_policy,
        primary_metric_name="val_accuracy",
        primary_metric_goal=PrimaryMetricGoal.MAXIMIZE,
        max_total_runs=12,  # 3 lr × 2 batch × 2 warmup = 12 runs
        max_concurrent_runs=4  # Run 4 in parallel (matches cluster size)
    )
    
    print("✓ HyperDrive configuration ready")
    print(f"  Search space: {param_sampling}")
    print(f"  Total runs: 12 (3×2×2 grid)")
    print(f"  Max concurrent: 4")
    print(f"  Compute: {AZURE_COMPUTE_VM_SIZE} (${AZURE_COMPUTE_VM_SIZE == 'Standard_NC6s_v3' and '3.06' or '?'}/hr per node)")
    print(f"\n💡 To execute: experiment.submit(hyperdrive_config)")
    print(f"   Estimated cost: ~$12 for 4-hour sweep (4 nodes × 1 hour)")
    
else:
    print("Demo mode: HyperDrive configuration (mock)")
    print("\nParameter search space:")
    print("  learning_rate: [1e-5, 3e-5, 5e-5]")
    print("  batch_size: [16, 32]")
    print("  warmup_steps: [100, 500]")
    print("  Total: 3 × 2 × 2 = 12 runs")
    print("\nEarly termination: BanditPolicy (slack_factor=0.1)")
    print("  → Stops runs if accuracy < 90% of best run")
    print("\n💡 In live mode, this would launch a distributed sweep on Azure ML compute.")
    print("   Results would appear in Azure ML Studio under Experiments → HyperDrive jobs.")

## 3 · Azure ML Model Registry

**Model Registry** = Centralized model storage with versioning and metadata

**Key differences from MLflow Model Registry:**
- Models are stored in Azure Blob Storage (workspace storage account)
- **Versioning:** Automatic (each registration creates a new version)
- **Stages:** Azure ML uses *tags* instead of stages (e.g., `stage=production`)
- **Deployment:** One-click deploy to Azure ML endpoints, ACI, AKS

**Workflow:**
1. **Register model** from run artifacts (e.g., `model.pt` from best HyperDrive run)
2. **Tag model** with metadata (e.g., `stage=staging`, `accuracy=0.94`)
3. **Download model** for inference (or deploy to Azure ML endpoint)
4. **Promote model** by updating tags (e.g., `stage=production`)

**Running example:** Register best model from HyperDrive sweep.

In [ ]:
# ── Register Model from Experiment Run ───────────────────────────────────────
# Simulate registering the best model from HyperDrive sweep

if USE_LIVE_API:
    # In real scenario, you'd get the best run from HyperDrive:
    # best_run = hyperdrive_run.get_best_run_by_primary_metric()
    # For demo, use the baseline run from Cell 3
    
    model = Model.register(
        workspace=ws,
        model_name="bert-sentiment-classifier",
        model_path="outputs/model.pt",  # Path in run artifacts
        model_framework=Model.Framework.PYTORCH,
        model_framework_version="2.1.0",
        tags={
            "stage": "staging",
            "accuracy": "0.94",
            "f1_score": "0.93",
            "training_date": datetime.now().strftime("%Y-%m-%d"),
            "use_case": "sentiment-analysis"
        },
        description="BERT fine-tuned on IMDB sentiment (best HyperDrive run)"
    )
    
    print(f"✓ Registered model: {model.name} (version {model.version})")
    print(f"  ID: {model.id}")
    print(f"  Tags: {model.tags}")
    print(f"  Storage: {model.url}")
    print(f"\n💡 View in Azure ML Studio: Models → {model.name}")
    
else:
    print("Demo mode: Model registered (mock)")
    print("\nModel details:")
    print("  Name: bert-sentiment-classifier")
    print("  Version: 1")
    print("  Framework: PyTorch 2.1.0")
    print("  Tags:")
    print("    stage: staging")
    print("    accuracy: 0.94")
    print("    f1_score: 0.93")
    print("\n💡 In live mode, model would be stored in workspace blob storage.")
    print("   Download with: Model(ws, 'bert-sentiment-classifier').download(target_dir='.')")

In [ ]:
# ── Download Model from Registry for Inference ───────────────────────────────
# Fetch model by name + version (or latest) and download for local inference

if USE_LIVE_API:
    # Get latest version of model
    model = Model(ws, name="bert-sentiment-classifier")
    
    # Download to local directory
    model_path = model.download(target_dir="./downloaded_models", exist_ok=True)
    
    print(f"✓ Downloaded model: {model.name} v{model.version}")
    print(f"  Local path: {model_path}")
    print(f"  Tags: {model.tags}")
    print(f"\n💡 Load with PyTorch:")
    print(f"   import torch")
    print(f"   model = torch.load('{model_path}/model.pt')")
    print(f"   model.eval()")
    
else:
    print("Demo mode: Model download (mock)")
    print("\nDownloaded:")
    print("  bert-sentiment-classifier v1 → ./downloaded_models/model.pt")
    print("\n💡 In live mode, this would download from Azure Blob Storage.")
    print("   Typical size: 440MB for BERT-base (110M parameters × 4 bytes)")

## 4 · Azure ML vs. MLflow UI Comparison

**Side-by-side feature comparison:**

| Feature | MLflow (Local) | Azure ML Studio |
|---------|----------------|------------------|
| **Experiment tracking** | ✅ Local SQLite DB | ✅ Cloud (blob storage) |
| **Model registry** | ✅ Local filesystem | ✅ Cloud (versioned) |
| **UI access** | localhost:5000 | portal.azure.com |
| **Parallel coordinates** | ✅ Built-in | ✅ HyperDrive viewer |
| **Artifact storage** | Local `mlruns/` | Azure Blob Storage |
| **Cost** | $0 (local compute) | Compute charges apply |
| **Collaboration** | Manual (shared DB) | Built-in (multi-user) |
| **Deployment** | Manual (Flask/FastAPI) | One-click (AML endpoints) |
| **Monitoring** | None | Azure Monitor integration |
| **Distributed training** | Manual (Ray/Dask) | Built-in (HyperDrive) |

**When to use MLflow:**
- Prototyping, local development
- No cloud budget
- Small team, simple workflows

**When to use Azure ML:**
- Production deployments
- Large-scale hyperparameter sweeps (>10 runs)
- Multi-user collaboration
- Need built-in monitoring and alerts

**💡 Pro tip:** Use MLflow locally, then migrate to Azure ML for production:
```python
# Local development with MLflow
mlflow.start_run()
mlflow.log_param("learning_rate", 5e-5)
mlflow.log_metric("accuracy", 0.94)
mlflow.pytorch.log_model(model, "model")

# Production deployment with Azure ML
run = experiment.start_logging()
run.log("learning_rate", 5e-5)
run.log("accuracy", 0.94)
run.upload_file("outputs/model.pt", "./model.pt")
Model.register(ws, "bert-model", model_path="outputs/model.pt")
```

## 5 · Cost Tracking for Experiments

**Cost breakdown for Azure ML experiments:**

**Compute costs (primary driver):**
- **VM instance:** Billed per second while running
- **Idle time:** No charges if cluster scales to 0 nodes
- **Storage:** ~$0.02/GB/month (minimal for model artifacts)

**Example costs (April 2026 pricing, East US):**
- `Standard_NC6s_v3` (1×V100): $3.06/hr
- `Standard_NC24s_v3` (4×V100): $12.24/hr
- `Standard_NC24ads_A100_v4` (1×A100): $5.50/hr

**Running example cost estimate:**
- **Single experiment:** 20 min × $3.06/hr = **$1.02**
- **HyperDrive sweep (12 runs, parallel):** 1 hr × $3.06 × 4 nodes = **$12.24**
- **HyperDrive sweep (12 runs, sequential):** 4 hrs × $3.06 × 1 node = **$12.24** (same cost, slower)

**Cost optimization tips:**
1. **Scale to zero:** Set `min_nodes=0` in cluster config
2. **Early termination:** Use BanditPolicy to stop bad runs
3. **Spot instances:** Save 60-80% with low-priority VMs (may be preempted)
4. **Smaller VMs:** Use `NC4as_T4_v3` ($0.53/hr) for prototyping
5. **Monitoring:** Set up budget alerts in Azure Cost Management

Below we simulate a cost tracking dashboard.

In [ ]:
# ── Cost Tracking Dashboard (Mock Data) ──────────────────────────────────────
# Simulate cost tracking for HyperDrive sweep + baseline experiments

# Mock experiment costs
experiment_costs = [
    {"run_id": "baseline-001", "vm_type": "NC6s_v3", "duration_min": 18, "cost_usd": 0.92},
    {"run_id": "hyperdrive-001", "vm_type": "NC6s_v3", "duration_min": 22, "cost_usd": 1.12},
    {"run_id": "hyperdrive-002", "vm_type": "NC6s_v3", "duration_min": 19, "cost_usd": 0.97},
    {"run_id": "hyperdrive-003", "vm_type": "NC6s_v3", "duration_min": 15, "cost_usd": 0.77},
    {"run_id": "hyperdrive-004", "vm_type": "NC6s_v3", "duration_min": 21, "cost_usd": 1.07},
    {"run_id": "hyperdrive-005", "vm_type": "NC6s_v3", "duration_min": 20, "cost_usd": 1.02},
    {"run_id": "hyperdrive-006", "vm_type": "NC6s_v3", "duration_min": 8, "cost_usd": 0.41},   # Early termination
    {"run_id": "hyperdrive-007", "vm_type": "NC6s_v3", "duration_min": 23, "cost_usd": 1.17},
    {"run_id": "hyperdrive-008", "vm_type": "NC6s_v3", "duration_min": 19, "cost_usd": 0.97},
    {"run_id": "hyperdrive-009", "vm_type": "NC6s_v3", "duration_min": 12, "cost_usd": 0.61},   # Early termination
    {"run_id": "hyperdrive-010", "vm_type": "NC6s_v3", "duration_min": 21, "cost_usd": 1.07},
    {"run_id": "hyperdrive-011", "vm_type": "NC6s_v3", "duration_min": 20, "cost_usd": 1.02},
    {"run_id": "hyperdrive-012", "vm_type": "NC6s_v3", "duration_min": 18, "cost_usd": 0.92},
]

cost_df = pd.DataFrame(experiment_costs)

# Summary statistics
total_cost = cost_df["cost_usd"].sum()
total_duration_hrs = cost_df["duration_min"].sum() / 60
avg_cost_per_run = cost_df["cost_usd"].mean()

print("Azure ML Experiment Cost Summary")
print("=" * 50)
print(f"Total experiments: {len(cost_df)}")
print(f"Total compute time: {total_duration_hrs:.1f} hours")
print(f"Total cost: ${total_cost:.2f}")
print(f"Average cost per run: ${avg_cost_per_run:.2f}")
print(f"\nCost savings from early termination:")
early_terminated = cost_df[cost_df["duration_min"] < 15]
print(f"  {len(early_terminated)} runs terminated early")
print(f"  Estimated savings: ${(20 - early_terminated['duration_min'].mean()) * 3.06 / 60 * len(early_terminated):.2f}")

# Cost breakdown visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Cost per run
ax = axes[0]
colors = ['#1f77b4' if 'baseline' in r else '#ff7f0e' for r in cost_df['run_id']]
bars = ax.bar(range(len(cost_df)), cost_df['cost_usd'], color=colors, alpha=0.7, edgecolor='black', linewidth=0.5)
ax.axhline(avg_cost_per_run, color='red', linestyle='--', linewidth=1.5, label=f'Avg: ${avg_cost_per_run:.2f}')
ax.set_xlabel('Run Number', fontsize=12, fontweight='bold')
ax.set_ylabel('Cost (USD)', fontsize=12, fontweight='bold')
ax.set_title('Cost per Experiment Run', fontsize=13, fontweight='bold')
ax.legend(loc='upper right')
ax.grid(axis='y', alpha=0.3)

# Plot 2: Duration vs. cost
ax = axes[1]
scatter_colors = ['#1f77b4' if 'baseline' in r else '#ff7f0e' for r in cost_df['run_id']]
ax.scatter(cost_df['duration_min'], cost_df['cost_usd'], c=scatter_colors, s=100, alpha=0.7, edgecolor='black', linewidth=0.5)
# Add trendline
z = np.polyfit(cost_df['duration_min'], cost_df['cost_usd'], 1)
p = np.poly1d(z)
ax.plot(cost_df['duration_min'], p(cost_df['duration_min']), "r--", linewidth=2, alpha=0.8, label=f'Trend: ${z[0]:.3f}/min')
ax.set_xlabel('Duration (minutes)', fontsize=12, fontweight='bold')
ax.set_ylabel('Cost (USD)', fontsize=12, fontweight='bold')
ax.set_title('Duration vs. Cost', fontsize=13, fontweight='bold')
ax.legend(loc='upper left')
ax.grid(alpha=0.3)

# Legend for colors
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#1f77b4', edgecolor='black', label='Baseline'),
    Patch(facecolor='#ff7f0e', edgecolor='black', label='HyperDrive')
]
axes[0].legend(handles=legend_elements + [plt.Line2D([0], [0], color='red', linestyle='--', label=f'Avg: ${avg_cost_per_run:.2f}')], loc='upper right')

plt.tight_layout()
plt.show()

print(f"\n💡 Cost optimization insights:")
print(f"   • Early termination saved ~${(20 * len(early_terminated) - early_terminated['duration_min'].sum()) * 3.06 / 60:.2f}")
print(f"   • Linear cost: ~${z[0]:.3f} per minute (matches ${3.06/60:.3f}/min for NC6s_v3)")
print(f"   • Parallel execution: 4 nodes × 1 hr = ${3.06 * 4:.2f} vs. 1 node × 4 hrs = ${3.06 * 4:.2f} (same cost)")

## Summary & Next Steps

**What you learned:**
1. ✅ Azure ML workspace connection and experiment logging
2. ✅ HyperDrive configuration for distributed hyperparameter sweeps
3. ✅ Model registry workflow (register, version, download)
4. ✅ Cost tracking and optimization strategies
5. ✅ Azure ML vs. MLflow comparison

**Key takeaways:**
- **Local MLflow** for prototyping (free, fast iteration)
- **Azure ML** for production (scalable, collaborative, monitored)
- **Cost optimization:** Scale to zero + early termination = 40-60% savings
- **Model registry** enables reproducibility (version + metadata + artifacts)

**Next steps:**
1. **Deploy model to Azure ML endpoint** (see Ch.10 for inference servers)
2. **Set up monitoring** (data drift, prediction drift, performance)
3. **A/B test models** (v1 vs. v2 in production)
4. **Automate retraining** (Azure ML pipelines for CI/CD)

**Resources:**
- [Azure ML Documentation](https://docs.microsoft.com/en-us/azure/machine-learning/)
- [HyperDrive Guide](https://docs.microsoft.com/en-us/azure/machine-learning/how-to-tune-hyperparameters)
- [Model Registry Best Practices](https://docs.microsoft.com/en-us/azure/machine-learning/how-to-manage-models)
- [Azure ML Pricing Calculator](https://azure.microsoft.com/en-us/pricing/calculator/)